In [1]:
# Imports
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

In [2]:
# Define DIRECTORY_PATH and TEST_PATH
DIRECTORY_PATH = Path("sample_input")
TEST_PATH = Path("sample_input/test_input")

In [3]:
def get_components(file_path):
    # Set up an adjacency list
    n = None
    graph = defaultdict(set)

    with open(file_path, 'r', encoding='utf-8') as file:
        for line in file:
            if not line.strip():
                continue
            if not n:
                n = int(line)
                if n == 0:
                    return n, 0, {}
                continue

            u, v = line.split()
            graph[int(u)].add(int(v))
            graph[int(v)].add(int(u))


    # Count connected components
    visited = [False] * n
    counter = 0
    degrees = defaultdict(int)

    def dfs(node):
        stk = [node]
        while len(stk) > 0:
            curr = stk.pop()

            for neighbor in graph[curr]:
                if not visited[neighbor]:
                    stk.append(neighbor)
                    visited[neighbor] = True

    for node in range(n):
        degrees[len(graph[node])] += 1
        if not visited[node]:
            visited[node] = True
            counter += 1
            dfs(node)

    return n, counter, dict(sorted(degrees.items()))

In [4]:
# Check logic on test files
tests = 0
for file_path in sorted(TEST_PATH.iterdir()):
    if file_path.is_file():
        tests += 1
        print(f"File Name: {file_path.stem}")
        n, components, degrees = get_components(file_path)
        # Check that total degrees == N
        assert sum(degrees.values()) == n

        if file_path.stem == "t3" or file_path.stem == "t5":
            assert components == 3
        elif file_path.stem == "t4":
            assert components == 1

assert tests > 0
print("All checks passed!")

File Name: t3
File Name: t4
File Name: t5
All checks passed!


In [5]:
# Loop through all the files and get info
Path("plots").mkdir(exist_ok=True)

for file_path in sorted(DIRECTORY_PATH.iterdir()):
    if file_path.is_file():
        # Analyze this file
        print(f"File Name: {file_path.stem}")
        n, components, degrees = get_components(file_path)

        # If there's no nodes, skip the plot
        if n == 0:
            print("Graph is empty")
            continue
        # Check that total degrees == N
        assert sum(degrees.values()) == n

        # Build the x and y values
        x, y = [], []
        max_deg = max(degrees)
        for deg in range(max_deg + 1):
            x.append(deg)
            y.append(degrees.get(deg, 0))

        # Check that x and y have been built correctly
        assert len(x) == len(y)
        assert len(x) == max_deg + 1
        assert sum(y) == n

        # Create the chart
        plt.bar(x, y)
        ax = plt.gca()
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))

        # Labels
        plt.title(f"{file_path.stem}: N={n}, components={components}")
        plt.xlabel("Degree")
        plt.ylabel("# of Nodes")

        # Save the chart and refresh for the next graph
        plt.savefig(f"plots/{file_path.stem}.png")
        plt.close()

        # Print component count
        print(f"# of Components: {components}")
        print()

File Name: n10
# of Components: 1

File Name: n100
# of Components: 12

File Name: n1000
# of Components: 17

File Name: n10000
# of Components: 12

File Name: s1
# of Components: 6

